# Minggu7 / Sub-CPMK7 · Dari piksel ke objek
## Notebook MAHASISWA: thresholding,HSV,dan Watershed

**Digitalisasi arsip · hitung koin · seleksi warna buah**

Otsu,Adaptive Mean,Adaptive Gaussian,segmentasi HSV,marker-based Watershed. Sel inti sama pada kedua notebook. Versi pengajar menyimpan output serta contoh pengamatan; versi mahasiswa disediakan dengan output kosong.

**Nama: … | NIM: … | Kelas: … | Tanggal: …**

Pelajaran150 menit: teori50,demo50,replay50. Notebook mendukung demonstrasi satu arah dan praktikum terpandu. Tidak memerlukan polling/diskusi kelas.

## Rute menjalankan notebook

1. Buka [Google Colab](https://colab.research.google.com/) → **File → Upload notebook** lalu pilih berkas ini.
2. Gunakan runtime Python 3 **CPU**. Simpan salinan di Drive bila diperlukan.
3. Jalankan dari **A01 ke E02**, atau pilih **Runtime → Run all**. Eksperimen opsional akan dilewati sampai saklarnya diubah.
4. Setelah selesai, simpan notebook beserta output. Tugas inti memakai gambar dan parameter yang sama dengan demonstrasi.

**Waktu replay 50 menit:** setup 4; dokumen 10; koin 20; HSV 8; pengamatan 5; pengumpulan 3.

Kode wajib sudah lengkap. Fokuskan waktu pada menjalankan sel, membaca gambar antara, dan menulis pengamatan. Unduhan paket Python hanya terjadi bila import belum tersedia; input gambar inti tidak memerlukan jaringan. Notebook pengajar menyimpan hasil eksekusi, sehingga dapat menjadi cadangan bila jaringan kelas terganggu.

### A01 · Lingkungan praktikum
Keluaran yang diharapkan: versi OpenCV/NumPy dan pesan CPU siap. Versi lokal untuk QA dicatat di laporan; versi Colab dapat berbeda. Bila satu sel gagal, perbaiki pesan kesalahan lalu jalankan ulang dari A01.

In [ ]:
# A01 — Periksa lingkungan. CPU cukup; tidak perlu GPU.
import importlib.util, subprocess, sys
packages = {'cv2': 'opencv-python-headless', 'numpy': 'numpy', 'matplotlib': 'matplotlib'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import cv2, numpy as np, matplotlib.pyplot as plt
import base64, hashlib, json
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 14, 'figure.dpi': 100})
print('OpenCV:', cv2.__version__, '| NumPy:', np.__version__, '| CPU siap')

### A02 · Fungsi bantu dan parameter inti
Sel ini mendefinisikan fungsi, sehingga belum menghasilkan gambar. Nilai tetap pada tugas inti: **dokumen B=31,C=9; seed ratio=0.50; luas minimum koin=50px**. Rentang HSV berasal dari pilot pada foto pilihan. Perubahan parameter ditempatkan pada F01 agar hasil inti tetap dapat dibandingkan.

Fungsi `watershed_coins` memperlihatkan pipeline lengkap. Bacalah terutama bagian `envelope`, `unknown`, `markers_before`, dan `valid_ids`: latar pasti bernilai1, unknown0, seed objek≥2, batas hasil−1.

In [ ]:
# A02 — Fungsi bantu yang dipakai pada seluruh kasus.
"""Functions shared verbatim by both teaching notebooks; no external asset reads."""
import cv2
import numpy as np
import matplotlib.pyplot as plt

DOC_BLOCK = 31
DOC_C = 9
DOC_ROI = (65, 110, 545, 355)  # x0,y0,x1,y1 on the curated 600x969 page
SEED_RATIO = 0.50
MIN_COIN_AREA = 50
HSV_RANGES = {
    'red_low': ([0, 80, 45], [10, 255, 255]),
    'red_high': ([165, 80, 45], [179, 255, 255]),
    'orange': ([13, 80, 45], [28, 255, 255]),
}

def rgb(bgr):
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def panels(items, cols=3, figsize=None):
    """items: (title,array,cmap); BGR images must be converted with rgb()."""
    rows = (len(items) + cols - 1) // cols
    if figsize is None:
        figsize = (4.4 * cols, 4.1 * rows)
    fig, axes = plt.subplots(rows, cols, figsize=figsize, squeeze=False)
    for ax, item in zip(axes.flat, items):
        title, arr, cmap = item
        ax.imshow(arr, cmap=cmap, interpolation='nearest')
        ax.set_title(title, fontsize=16, pad=10)
        ax.axis('off')
    for ax in list(axes.flat)[len(items):]:
        ax.axis('off')
    fig.tight_layout(pad=1.5)
    return fig

def document_masks(doc_gray, block=DOC_BLOCK, c=DOC_C):
    if doc_gray.dtype != np.uint8 or doc_gray.ndim != 2:
        raise ValueError('Dokumen harus grayscale uint8.')
    if block <= 1 or block % 2 != 1 or block > min(doc_gray.shape):
        raise ValueError('blockSize harus ganjil, >1, dan tidak melebihi sisi terpendek.')
    threshold, otsu = cv2.threshold(doc_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mean = cv2.adaptiveThreshold(doc_gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, block, c)
    gaussian = cv2.adaptiveThreshold(doc_gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, block, c)
    return dict(otsu=otsu, mean=mean, gaussian=gaussian, threshold=float(threshold), block=int(block), c=int(c))

def watershed_coins(coins_bgr, ratio=SEED_RATIO, min_area=MIN_COIN_AREA):
    if coins_bgr is None or coins_bgr.dtype != np.uint8 or coins_bgr.ndim != 3:
        raise ValueError('Input harus gambar BGR uint8 tiga kanal.')
    if not 0 < ratio < 1:
        raise ValueError('Rasio harus di antara 0 dan 1.')
    gray = cv2.cvtColor(coins_bgr, cv2.COLOR_BGR2GRAY)
    threshold, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel = np.ones((3, 3), np.uint8)
    clean = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)
    distance = cv2.distanceTransform(clean, cv2.DIST_L2, 5)
    _, sure_fg_float = cv2.threshold(distance, ratio * distance.max(), 255, cv2.THRESH_BINARY)
    sure_fg = sure_fg_float.astype(np.uint8)
    # Envelope putih mencakup objek + margin; LATAR PASTI berada DI LUARNYA.
    envelope = cv2.dilate(clean, kernel, iterations=3)
    unknown = cv2.subtract(envelope, sure_fg)
    n_seed_labels, seeds = cv2.connectedComponents(sure_fg, connectivity=8)
    markers_before = (seeds + 1).astype(np.int32)
    markers_before[unknown == 255] = 0
    assert np.all(markers_before[envelope == 0] == 1)
    assert np.all(markers_before[unknown == 255] == 0)
    assert np.all(markers_before[sure_fg > 0] >= 2)
    labels = cv2.watershed(coins_bgr.copy(), markers_before.copy())
    # Count dari ID instance; bukan max(label), bukan CCL ulang gabungan mask.
    valid_ids = [int(i) for i in np.unique(labels) if i >= 2 and np.count_nonzero(labels == i) >= min_area]
    areas = {str(i): int(np.count_nonzero(labels == i)) for i in valid_ids}
    edge_ids = [i for i in valid_ids if any(np.any(edge == i) for edge in [labels[1,:], labels[-2,:], labels[:,1], labels[:,-2]])]
    overlay = coins_bgr.copy()
    overlay[labels == -1] = [0, 60, 255]
    for number, label_id in enumerate(valid_ids, 1):
        ys, xs = np.where(labels == label_id)
        center = (int(xs.mean()), int(ys.mean()))
        cv2.putText(overlay, str(number), center, cv2.FONT_HERSHEY_SIMPLEX, .36, (255,255,255), 2, cv2.LINE_AA)
        cv2.putText(overlay, str(number), center, cv2.FONT_HERSHEY_SIMPLEX, .36, (30,30,30), 1, cv2.LINE_AA)
    return dict(gray=gray, mask=mask, clean=clean, distance=distance, sure_fg=sure_fg,
        envelope=envelope, unknown=unknown, seeds=seeds, markers_before=markers_before, labels=labels,
        overlay=overlay, valid_ids=valid_ids, count=len(valid_ids), seed_count=n_seed_labels-1,
        initial_components=cv2.connectedComponents(mask, connectivity=8)[0]-1,
        clean_components=cv2.connectedComponents(clean, connectivity=8)[0]-1,
        threshold=float(threshold), ratio=float(ratio), min_area=int(min_area), areas=areas, edge_ids=edge_ids)

def hsv_select(fruit_bgr):
    if fruit_bgr is None or fruit_bgr.dtype != np.uint8 or fruit_bgr.ndim != 3:
        raise ValueError('Input harus gambar BGR uint8 tiga kanal.')
    hsv = cv2.cvtColor(fruit_bgr, cv2.COLOR_BGR2HSV)
    masks = {name: cv2.inRange(hsv, np.array(lo, np.uint8), np.array(hi, np.uint8))
             for name, (lo,hi) in HSV_RANGES.items()}
    red = cv2.bitwise_or(masks['red_low'], masks['red_high'])
    orange = masks['orange']
    overlay = fruit_bgr.copy()
    boxes = {}
    for name, mask, color in [('merah', red, (150,30,225)), ('oranye', orange, (0,150,255))]:
        n, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)
        box_records = []
        for i in range(1,n):
            x,y,w,h,area = map(int,stats[i])
            if area >= 200:
                cv2.rectangle(overlay,(x,y),(x+w-1,y+h-1),color,2)
                cv2.putText(overlay,name,(x,max(18,y-6)),cv2.FONT_HERSHEY_SIMPLEX,.5,color,1,cv2.LINE_AA)
                box_records.append(dict(x=x,y=y,w=w,h=h,area=area))
        boxes[name] = box_records
    return dict(hsv=hsv, red=red, orange=orange, red_low=masks['red_low'], red_high=masks['red_high'],
        overlay=overlay, boxes=boxes, red_pixels=int(np.count_nonzero(red)), orange_pixels=int(np.count_nonzero(orange)))

def marker_view(markers):
    """Color known bg blue, unknown gray, seeds categorical; actual values retained."""
    h,w = markers.shape
    view = np.full((h,w,3), [36,74,118], np.uint8)
    view[markers == 0] = [225,226,230]
    palette = np.array([[236,87,102],[32,174,136],[252,182,51],[135,101,220],[32,159,207]],np.uint8)
    for i in np.unique(markers):
        if i >= 2:
            view[markers == i] = palette[(int(i)-2)%len(palette)]
    return view

### A03 · Input, checksum, dan provenance

| Kasus | Data | Perubahan yang dilakukan |
|---|---|---|
| A | Scan halaman judul *Pride and Prejudice*, 1813, Jane Austen / Lilly Library, Wikimedia Commons; public domain | Resize ke600×969, grayscale, lalu **bayangan digital deterministik**. Bayangan merupakan simulasi pada scan nyata. |
| B | Foto `water_coins.jpg`, tutorial resmi OpenCV, tag4.13.0;312×252 | Decode JPEG → PNG lossless; tanpa crop/resize. LICENSE Apache-2.0 dan COPYRIGHT repository disertakan. |
| C | Foto apel merah Sprinno(CC0) dan jeruk AndriusR./Knutux(public domain), Commons | Resize dua foto lalu tempel pada kanvas putih552×290. **Komposisi digital dua foto nyata**, bukan rekaman konveyor. |

Sel data yang panjang boleh dilipat. Raw input telah ditanam ke notebook sebagai base64, dan checksum SHA-256 diperiksa sebelum digunakan. Tidak ada mask ground truth pada tiga kasus ini. Pemeriksaan dokumen/warna menggunakan gambar; count koin memakai referensi visual24, bukan skor akurasi.

In [ ]:
# A03 — Data kecil tertanam. Tidak mengunduh gambar ketika praktikum.
# Sel ini panjang karena berisi data; jalankan sekali, tidak perlu diedit.
ASSETS = {'document_base': {'name': 'document_base.png', 'sha256': '9868032b7751187e1888048c19fb392c50c856fb1461d229d6638889ceddf8e1', 'base64': '...'}, 'document': {'name': 'document_shadow.png', 'sha256': '35877cfd0182e4c3fdebce65f3d57ae67a24b20f6ea92843ed9e5709cc5de34e', 'base64': '...'}, 'coins': {'name': 'coins.png', 'sha256': '452ed66d53505ee8e3c7b86f2a4c413915d1ff47a98fcf3ba19a8eb21726b101', 'base64': '...'}, 'fruit': {'name': 'fruit_composite.png', 'sha256': '5e0d64232cfa2a2305b37e19eb9722188e30049b10af82693f8cfe7bddea1905', 'base64': '...'}}
def decode_asset(key, grayscale=False):
    raw = base64.b64decode(ASSETS[key]['base64'])
    assert hashlib.sha256(raw).hexdigest() == ASSETS[key]['sha256'], 'Checksum data tidak cocok.'
    flag = cv2.IMREAD_GRAYSCALE if grayscale else cv2.IMREAD_COLOR
    img = cv2.imdecode(np.frombuffer(raw, np.uint8), flag)
    if img is None:
        raise ValueError('Data tidak berhasil dibaca.')
    return img
doc_base = decode_asset('document_base', True)
doc_gray = decode_asset('document', True)
coins_bgr = decode_asset('coins')
fruit_bgr = decode_asset('fruit')
for name, arr in [('scan', doc_gray), ('koin', coins_bgr), ('buah', fruit_bgr)]:
    print(name, 'shape=', arr.shape, 'dtype=', arr.dtype)
print('Seluruh checksum input terverifikasi. Data demonstrasi dan latihan identik.')

## Kasus A · Digitalisasi arsip: menjaga teks ketika pencahayaan berubah
Kebutuhan aplikasi: binarisasi sebagai salah satu langkah persiapan OCR atau arsip digital. Kita memakai scan nyata dengan **simulasi bayangan**, sehingga degradasi dapat diulang persis. Notebook ini tidak menjalankan OCR dan tidak menghitung akurasi OCR.

**Polarity:** `THRESH_BINARY` menampilkan tulisan gelap sebagai0/hitam dan latar sebagai255/putih. Makna foreground tidak selalu berarti piksel putih; di kasus B pilihan ini dibalik.

In [ ]:
# B01 — Bandingkan scan tanpa degradasi dan input dengan bayangan simulasi.
fig = panels([('Scan nyata / resize', doc_base, 'gray'),
              ('Bayangan digital / input inti', doc_gray, 'gray')], cols=2, figsize=(9, 7))
plt.show()

### B02 · Tiga metode pada input yang sama
Otsu memilih satu ambang global berdasarkan variansi antarkelas. Mean dan Gaussian membuat ambang lokal dengan aturan `T(x,y)=nilai_lokal−C`. **B dan C kedua adaptif dibuat sama**, agar perbandingan terarah. B=31 ganjil; C=9 mengikuti pilot, bukan angka universal.

Pada tampilan BINARY, menaikkan C menurunkan ambang dan cenderung membuat lebih banyak piksel putih. Huruf gelap dapat menipis atau hilang. Untuk mask invers, bahasa putih/hitam harus disesuaikan.

In [ ]:
# B02 — Semua metode menggunakan doc_gray yang sama.
doc_result = document_masks(doc_gray, DOC_BLOCK, DOC_C)
print(f"Otsu T={doc_result['threshold']:.0f}; kedua adaptif B={DOC_BLOCK}, C={DOC_C}")
fig = panels([('Input / bayangan simulasi', doc_gray, 'gray'),
              ('Otsu / global', doc_result['otsu'], 'gray'),
              ('Adaptive Mean', doc_result['mean'], 'gray'),
              ('Adaptive Gaussian', doc_result['gaussian'], 'gray')], cols=4, figsize=(15, 6))
plt.show()
fig, ax = plt.subplots(figsize=(10, 3.7))
ax.hist(doc_gray.ravel(), bins=256, range=(0, 256), color='#20a89b')
ax.axvline(doc_result['threshold'], color='#e84f75', linewidth=3, label=f"T Otsu={doc_result['threshold']:.0f}")
ax.set(xlabel='Intensitas grayscale', ylabel='Jumlah piksel')
ax.legend(); fig.tight_layout(); plt.show()

In [ ]:
# B03 — Crop pada koordinat yang sama, tanpa mengganti input/parameter.
x0, y0, x1, y1 = DOC_ROI
fig = panels([(title, arr[y0:y1, x0:x1], 'gray') for title, arr in [
    ('ROI input', doc_gray), ('ROI Otsu', doc_result['otsu']),
    ('ROI Mean', doc_result['mean']), ('ROI Gaussian', doc_result['gaussian'])]], cols=2, figsize=(13, 7))
plt.show()
print('ROI x0,y0,x1,y1 =', DOC_ROI)
print('Catat huruf yang hilang/terputus dan bintik latar. Hasil visual bukan skor OCR.')

## Kasus B · Menghitung koin yang bersentuhan
Kebutuhan aplikasi: menghitung benda pada inspeksi visual atau inventaris. Semua24 koin terlihat pada foto; beberapa bersentuhan. **Bersentuhan berbeda dari teroklusi:** Watershed tidak merekonstruksi bagian benda yang tertutup.

Referensi24 berasal dari penghitungan visual dua peninjau, dengan seluruh koin dekat tepi ikut dihitung. Titik pusat dipakai sebagai dokumentasi count, bukan anotasi bentuk/ground truth mask.

In [ ]:
# C01 — Tampilkan input dan referensi count yang didokumentasikan.
MANUAL_CENTERS = [[64, 39], [135, 53], [94, 74], [176, 86], [46, 99], [132, 103], [219, 105], [89, 121], [23, 138], [161, 139], [118, 156], [205, 156], [64, 163], [153, 192], [26, 193], [94, 199], [198, 202], [58, 229], [129, 234], [177, 246], [92, 265], [47, 280], [134, 292], [180, 297]]
MANUAL_COUNT = len(MANUAL_CENTERS)
ref_view = coins_bgr.copy()
for number, (cx, cy) in enumerate(MANUAL_CENTERS, 1):
    cv2.circle(ref_view, (cx, cy), 3, (0, 255, 255), -1)
    cv2.putText(ref_view, str(number), (cx-6, cy-8), cv2.FONT_HERSHEY_SIMPLEX, .33, (0,255,255), 1, cv2.LINE_AA)
fig = panels([('Foto koin asli', rgb(coins_bgr), None),
              (f'{MANUAL_COUNT} pusat / referensi visual', rgb(ref_view), None)], cols=2, figsize=(9, 5))
plt.show()

### C02 · Mask → seed → marker yang benar

1. Otsu invers menjadikan koin=255 dan latar=0.
2. Opening3×3 dua iterasi membersihkan piksel kecil; pada data lain dapat menghapus objek kecil.
3. Distance transform mengukur jarak foreground ke nol terdekat, dalam piksel.
4. Seed diambil dari distance>0.50×maksimum, lalu diberi ID komponen.
5. Envelope dibuat dengan dilasi tiga iterasi. **Bagian di luar envelope adalah latar yang pasti.**
6. Unknown adalah envelope dikurangi seed. Unknown mendapat0; latar1; seed objek≥2.
7. Watershed mengisi wilayah belum pasti dan menghasilkan batas−1.
8. Count menggunakan ID label objek yang valid; background,unknown,boundary tidak dihitung.

Kode `markers[clean==0]=0` dari contoh slide perlu diganti dengan pengosongan **unknown**. Sel berikut menampilkan ulang tiga baris pembentukan marker secara eksplisit dan memeriksa kesamaan dengan fungsi pipeline.

In [ ]:
# C02 — Jalankan pipeline. Parameter inti tidak diubah.
coin_result = watershed_coins(coins_bgr, SEED_RATIO, MIN_COIN_AREA)
# Tiga baris inti marker; sebelumnya sure_fg harus berupa mask uint8.
_, seed_labels = cv2.connectedComponents(coin_result['sure_fg'], connectivity=8)
marker_check = (seed_labels + 1).astype(np.int32)  # latar=1; seed>=2
marker_check[coin_result['unknown'] == 255] = 0  # hanya unknown yang nol
assert np.array_equal(marker_check, coin_result['markers_before'])
print('T Otsu =', coin_result['threshold'])
print('Komponen mask awal =', coin_result['initial_components'], '| setelah opening =', coin_result['clean_components'])
print('Seed =', coin_result['seed_count'], '| marker dtype =', coin_result['markers_before'].dtype)
print('Marker: background=1, unknown=0, seed objek>=2; boundary setelah Watershed=-1')

In [ ]:
# C03 — Delapan tahap wajib terlihat; warna categorical hanya alat visual.
fig = panels([
    ('1 Mask Otsu / objek putih', coin_result['mask'], 'gray'),
    ('2 Opening / cleaned mask', coin_result['clean'], 'gray'),
    ('3 Distance transform', coin_result['distance'], 'magma'),
    ('4 Sure foreground / seed', coin_result['sure_fg'], 'gray'),
    ('5 Envelope hasil dilasi', coin_result['envelope'], 'gray'),
    ('6 Unknown = envelope − seed', coin_result['unknown'], 'gray'),
    ('7 Marker sebelum Watershed', marker_view(coin_result['markers_before']), None),
    ('8 Label sesudah Watershed', coin_result['labels'], 'tab20')], cols=4, figsize=(15, 9))
plt.show()
print('Pada panel marker: biru=latar pasti1; abu-abu=unknown0; warna lain=seed ID>=2.')

### C04 · Count dan overlay harus dibaca bersama
Aturan tetap: ID≥2 dengan luas≥50px; label dekat tepi tetap ikut dihitung. Background1, unknown0, dan boundary−1 diabaikan. Mengambil `labels.max()` akan memasukkan offset label; melakukan CCL ulang pada union mask dapat menyatukan kembali objek.

Kecocokan24 dengan referensi visual bukan akurasi100%. Periksa apakah label memotong satu koin, menggabungkan dua koin, atau menutupi latar. Bentuk batas tidak dinilai kuantitatif karena mask referensi tidak tersedia.

In [ ]:
# C04 — Count memakai ID valid; ID tampil di atas sebagai nomor urutan1..N.
print('ID label valid:', coin_result['valid_ids'])
print('Luas minimum:', MIN_COIN_AREA, 'piksel; label dekat tepi tetap disertakan.')
print('Count hasil:', coin_result['count'], '| referensi visual:', MANUAL_COUNT)
print('Label yang dekat tepi:', coin_result['edge_ids'])
fig = panels([('Input nyata', rgb(coins_bgr), None),
              ('Label instance / warna dapat berulang', coin_result['labels'], 'tab20'),
              (f"Batas + {coin_result['count']} label valid", rgb(coin_result['overlay']), None)], cols=3, figsize=(12, 5))
plt.show()

In [ ]:
# C05 — Satu perbandingan seed yang sudah dihitung; hasil inti tidak ditimpa.
coin_comparison = watershed_coins(coins_bgr.copy(), ratio=0.35, min_area=MIN_COIN_AREA)
print('Rasio0.35:', coin_comparison['seed_count'], 'seed →', coin_comparison['count'], 'label valid')
print('Rasio0.50:', coin_result['seed_count'], 'seed →', coin_result['count'], 'label valid')
fig = panels([('0.35 / seed terlalu menyatu', coin_comparison['sure_fg'], 'gray'),
              ('0.35 / label hasil', coin_comparison['labels'], 'tab20'),
              ('0.50 / seed inti', coin_result['sure_fg'], 'gray'),
              ('0.50 / label inti', coin_result['labels'], 'tab20')], cols=4, figsize=(15, 5))
plt.show()
print('Rasio adalah parameter data, bukan jaminan satu seed per objek.')

## Kasus C · Seleksi warna untuk prototipe sortir buah
Kebutuhan aplikasi: memisahkan wilayah warna pada inspeksi produk. Input adalah **komposisi digital dua foto nyata**, bukan video sistem sortir yang diuji.

`cv2.imread/imdecode` menghasilkan BGR. Gunakan **COLOR_BGR2HSV**. Untuk input uint8 standard ini H0..179,S0..255,V0..255. Merah berada di dua ujung Hue, sehingga dua mask digabung. Ambang S membatasi piksel kurang berwarna; ambang V membatasi piksel sangat gelap. Hue pada piksel dengan S kecil tidak dapat dipakai sebagai bukti warna yang stabil.

Pilot memakai merah H0..10 atau165..179, oranye H13..28;S80..255,V45..255. Ada celah Hue11..12, sehingga semua piksel tidak wajib dipaksa menjadi salah satu warna. Rentang ini berbeda sedikit dari contoh slide karena diuji pada foto pilihan; tidak menjadi aturan universal buah.

In [ ]:
# D01 — Konversi BGR→HSV dan buat dua mask merah serta mask oranye.
fruit_result = hsv_select(fruit_bgr)
print('HSV uint8: H0..179; S/V0..255')
print('Rentang:', HSV_RANGES)
fig = panels([('Dua foto / komposisi digital', rgb(fruit_bgr), None),
              ('Merah bagian Hue rendah', fruit_result['red_low'], 'gray'),
              ('Merah bagian Hue tinggi', fruit_result['red_high'], 'gray'),
              ('Oranye', fruit_result['orange'], 'gray')], cols=2, figsize=(12, 7))
plt.show()

In [ ]:
# D02 — Mask harus diperiksa sebelum melihat kotak.
print('Piksel mask merah:', fruit_result['red_pixels'], '| oranye:', fruit_result['orange_pixels'])
print('Kotak region>=200px:', fruit_result['boxes'])
fig = panels([('Mask merah / gabungan dua Hue', fruit_result['red'], 'gray'),
              ('Mask oranye', fruit_result['orange'], 'gray'),
              ('Kotak REGION WARNA', rgb(fruit_result['overlay']), None)], cols=3, figsize=(15, 4.8))
plt.show()
print('Kotak menunjukkan geometri region warna; bukan confidence atau pengenal jenis buah umum.')
print('Area gelap apel dan sorotan terang dapat tidak masuk mask. Bintik kecil tetap terlihat pada mask.')

### E01 · Ringkasan hasil inti
Simpan keluaran ini bersama gambar B03,C03,C04,C05,D02. Angka piksel hanya mendeskripsikan mask pada data ini, bukan nilai kualitas. Tidak ada target akurasi100%.

In [ ]:
# E01 — Ringkasan yang dipakai untuk memeriksa konsistensi replay.
CORE_SUMMARY = {
    'document': {'T_otsu': doc_result['threshold'], 'blockSize': DOC_BLOCK, 'C': DOC_C,
                 'ROI': list(DOC_ROI), 'polarity': 'tulisan0; latar255'},
    'coins': {'components_before': coin_result['initial_components'], 'components_clean': coin_result['clean_components'],
              'seed_ratio': SEED_RATIO, 'seed_count': coin_result['seed_count'], 'valid_count': coin_result['count'],
              'manual_count': MANUAL_COUNT, 'min_area': MIN_COIN_AREA, 'valid_ids': coin_result['valid_ids'],
              'ratio035_seed_count': coin_comparison['seed_count'], 'ratio035_count': coin_comparison['count']},
    'hsv': {'ranges': HSV_RANGES, 'red_pixels': fruit_result['red_pixels'], 'orange_pixels': fruit_result['orange_pixels'],
            'red_boxes': len(fruit_result['boxes']['merah']), 'orange_boxes': len(fruit_result['boxes']['oranye'])},
    'input_sha256': {key: value['sha256'] for key, value in ASSETS.items()},
}
print(json.dumps(CORE_SUMMARY, indent=2, ensure_ascii=False))

### E02 · Pengamatan tertulis dan bukti pengumpulan
Isi **Nama: … | NIM: … | Kelas: … | Tanggal: …** pada bagian identitas di atas.

Tuliskan1–3 kalimat untuk setiap poin. Gunakan bagian gambar tertentu sebagai bukti, bukan hanya “bagus” atau “jelek”.

1. **Dokumen:** pada ROI B03, bagian huruf mana yang hilang di Otsu? Bandingkan bintik/latar dan goresan pada Mean serta Gaussian. **Jawaban:** …
2. **Parameter/polarity:** catat B,C,T Otsu; jelaskan arti0 dan255 pada mask dokumen serta mask koin. **Jawaban:** …
3. **Marker:** jelaskan latar pasti1,unknown0,seed≥2,batas−1 dengan menunjuk C03. **Jawaban:** …
4. **Count:** catat komponen awal,seed,label valid,referensi24. Jelaskan mengapa count cocok belum membuktikan kualitas bentuk/instance. **Jawaban:** …
5. **Seed:** bandingkan C05 rasio0.35 dan0.50; hubungkan seed menyatu dengan count hasil. **Jawaban:** …
6. **HSV:** tunjukkan area gelap/sorotan atau bintik yang tidak sesuai harapan; jelaskan mengapa kotak warna bukan pengenalan jenis buah umum. **Jawaban:** …

**Pengumpulan:** notebook `.ipynb` beserta output B03,C03,C04,C05,D02,E01 dan keenam pengamatan. Tidak perlu membuat pipeline baru. Rubrik: dokumen30%,Watershed40%,HSV20%,pengamatan10%. Eksperimen tambahan tidak menggantikan bukti inti.

## Eksperimen opsional · sesudah tugas inti
F01 mengubah satu nilaiC pada salinan input. F02 menerima gambar sendiri dan menjalankan salah satu pipeline pada variabel baru. Keduanya defaultFalse dan tidak menghalangiRunall. Catat data/parameter,lihatmask,dan jelaskan keterbatasan. Hasil eksperimen tidak menimpa `doc_result`,`coin_result`,`fruit_result`,atau `CORE_SUMMARY`.

In [ ]:
# F01 — OPSIONAL. Ubah True setelah tugas inti selesai.
RUN_PARAMETER_EXPERIMENT = False
EXPERIMENT_C = 13  # hanya satu parameter C; B dan input tetap
if RUN_PARAMETER_EXPERIMENT:
    optional_doc = document_masks(doc_gray.copy(), block=DOC_BLOCK, c=EXPERIMENT_C)
    ox0, oy0, ox1, oy1 = DOC_ROI
    fig = panels([('Gaussian inti / C=9', doc_result['gaussian'][oy0:oy1, ox0:ox1], 'gray'),
                  (f'Gaussian eksperimen / C={EXPERIMENT_C}', optional_doc['gaussian'][oy0:oy1, ox0:ox1], 'gray')], cols=2, figsize=(12, 4.7))
    plt.show()
    print('Hasil inti dan CORE_SUMMARY tetap utuh. Tulis pengaruh C pada huruf/bintik latar.')
else:
    print('F01 dilewati; tugas inti telah lengkap.')

In [ ]:
# F02 — OPSIONAL. Run all tidak memunculkan dialog upload secara otomatis.
RUN_UPLOAD = False
UPLOAD_MODE = 'document'  # pilihan: document, coins, color

def process_uploaded_bytes(raw_bytes, mode='document'):
    if not raw_bytes:
        raise ValueError('Berkas kosong. Pilih gambar JPG/PNG yang valid.')
    if len(raw_bytes) > 10 * 1024 * 1024:
        raise ValueError('Maksimum10MB. Perkecil gambar terlebih dahulu.')
    uploaded = cv2.imdecode(np.frombuffer(raw_bytes, np.uint8), cv2.IMREAD_COLOR)
    if uploaded is None:
        raise ValueError('Berkas tidak terbaca sebagai gambar. Gunakan JPG/PNG, bukan PDF/HEIC.')
    if min(uploaded.shape[:2]) < 32:
        raise ValueError('Gambar terlalu kecil; sisi terpendek minimal32px.')
    if max(uploaded.shape[:2]) > 1600:
        scale = 1600 / max(uploaded.shape[:2])
        uploaded = cv2.resize(uploaded, (round(uploaded.shape[1]*scale), round(uploaded.shape[0]*scale)), interpolation=cv2.INTER_AREA)
    if mode == 'document':
        uploaded_gray = cv2.cvtColor(uploaded, cv2.COLOR_BGR2GRAY)
        result = document_masks(uploaded_gray, block=DOC_BLOCK, c=DOC_C)
        fig = panels([('Input sendiri', rgb(uploaded), None), ('Otsu',result['otsu'],'gray'),
                      ('Mean',result['mean'],'gray'), ('Gaussian',result['gaussian'],'gray')], cols=2)
        message = 'Periksa polarity teks0/latar255 dan apakah B31 masih sesuai skala huruf.'
    elif mode == 'coins':
        result = watershed_coins(uploaded.copy(), ratio=SEED_RATIO, min_area=MIN_COIN_AREA)
        fig = panels([('Input sendiri',rgb(uploaded),None), ('Seed',result['sure_fg'],'gray'),
                      ('Hasil objek',rgb(result['overlay']),None)], cols=3)
        message = f"Count label={result['count']}. Asumsi koin gelap/latar terang harus diperiksa; ini bukan count terverifikasi."
    elif mode == 'color':
        result = hsv_select(uploaded.copy())
        fig = panels([('Input sendiri',rgb(uploaded),None), ('Mask merah',result['red'],'gray'),
                      ('Mask oranye',result['orange'],'gray'), ('Region warna',rgb(result['overlay']),None)], cols=2)
        message = 'Rentang inti mungkin tidak cocok dengan cahaya/warna kamera; periksa mask sebelum memberi label.'
    else:
        raise ValueError('UPLOAD_MODE harus document, coins, atau color.')
    return result, fig, message

if RUN_UPLOAD:
    try:
        from google.colab import files
    except ImportError:
        print('Dialog upload tersedia di Google Colab. Fungsi process_uploaded_bytes dapat dipanggil lokal dengan bytes JPG/PNG.')
    else:
        uploaded_files = files.upload()
        if not uploaded_files:
            print('Tidak ada berkas; hasil inti tetap utuh.')
        for optional_name, optional_bytes in uploaded_files.items():
            try:
                optional_result, optional_figure, optional_message = process_uploaded_bytes(optional_bytes, UPLOAD_MODE)
                print(optional_name, '|', optional_message); plt.show()
            except ValueError as err:
                print(optional_name, '|', err)
else:
    print('F02 dilewati. Aktifkan setelah tugas inti; unggah data yang boleh dibagikan dan hindari informasi pribadi.')

## Sumber belajar dan atribusi data

- [OpenCV: Thresholding](https://docs.opencv.org/4.x/d7/d4d/tutorial_py_thresholding.html) — Otsu,Mean,Gaussian.
- [OpenCV: Miscellaneous image transformations](https://docs.opencv.org/4.x/d7/d1b/group__imgproc__misc.html) — formula adaptive,threshold,distance transform.
- [OpenCV: Watershed](https://docs.opencv.org/4.x/d3/db4/tutorial_py_watershed.html) — envelope,unknown,marker,dan contoh pemisahan.
- [OpenCV: Segmentation API](https://docs.opencv.org/4.x/d3/d47/group__imgproc__segmentation.html) — nilai marker dan boundary.
- [OpenCV: Colorspaces](https://docs.opencv.org/4.x/df/d9d/tutorial_py_colorspaces.html) — BGR→HSV,skala uint8.
- [Scan Pride and Prejudice1813](https://commons.wikimedia.org/wiki/File:PrideAndPrejudiceTitlePage.jpg) — Jane Austen; arsip Lilly Library; upload Commons Awadewit; public domain/PDM. File sumber sudah mengandung edit penghapusan tanda tangan menurut riwayat Commons. Paket mengubah ukuran dan menambahkan bayangan simulasi.
- [Foto water_coins OpenCV4.13.0](https://github.com/opencv/opencv/blob/4.13.0/doc/py_tutorials/py_imgproc/py_watershed/images/water_coins.jpg) — OpenCV project, default repository Apache-2.0; tidak ditemukan pemberitahuan lisensi khusus pada direktori. Foto tidak dicrop/resize; decode disimpan PNG; hasil analisis/overlay dibuat pada paket ini.
- [Foto apel merah](https://commons.wikimedia.org/wiki/File:Red_apple_on_white_background.jpg) — Sprinno,CC0 1.0; resize dan komposisi digital.
- [Foto jeruk](https://commons.wikimedia.org/wiki/File:Orange-fruit.jpg) — AndriusR./Knutux,author release public domain; resize dan komposisi digital.
- [Metrics Reloaded](https://arxiv.org/html/2206.01653v8) — membedakan penilaian count,instance,dan mask sesuai tujuan.

Sumber diperiksa7 Oktober2026. Manifest paket mencatat SHA-256 sumber/kurasi dan transformasi. Notebook inti tidak memakai ilustrasiAI sebagai data. Uji lokal dari kernel bersih tidak sama dengan uji liveGoogleColab atau validasi waktu kelas.

<details><summary>LICENSE dan COPYRIGHT repository OpenCV (salinan lengkap untuk distribusi notebook)</summary>

```text
Copyright (C) 2000-2022, Intel Corporation, all rights reserved.
Copyright (C) 2009-2011, Willow Garage Inc., all rights reserved.
Copyright (C) 2009-2016, NVIDIA Corporation, all rights reserved.
Copyright (C) 2010-2013, Advanced Micro Devices, Inc., all rights reserved.
Copyright (C) 2015-2023, OpenCV Foundation, all rights reserved.
Copyright (C) 2008-2016, Itseez Inc., all rights reserved.
Copyright (C) 2019-2023, Xperience AI, all rights reserved.
Copyright (C) 2019-2022, Shenzhen Institute of Artificial Intelligence and Robotics for Society, all rights reserved.
Copyright (C) 2022-2023, Southern University of Science And Technology, all rights reserved.
Copyright (C) 2023-2025, OpenCV AI, all rights reserved.

Third party copyrights are property of their respective owners.


                                 Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by contract or
      otherwise, or (ii) ownership of fifty percent (50%) or more of the
      outstanding shares, or (iii) beneficial ownership of such entity.

      "You" (or "Your") shall mean an individual or Legal Entity
      exercising permissions granted by this License.

      "Source" form shall mean the preferred form for making modifications,
      including but not limited to software source code, documentation
      source, and configuration files.

      "Object" form shall mean any form resulting from mechanical
      transformation or translation of a Source form, including but
      not limited to compiled object code, generated documentation,
      and conversions to other media types.

      "Work" shall mean the work of authorship, whether in Source or
      Object form, made available under the License, as indicated by a
      copyright notice that is included in or attached to the work
      (an example is provided in the Appendix below).

      "Derivative Works" shall mean any work, whether in Source or Object
      form, that is based on (or derived from) the Work and for which the
      editorial revisions, annotations, elaborations, or other modifications
      represent, as a whole, an original work of authorship. For the purposes
      of this License, Derivative Works shall not include works that remain
      separable from, or merely link (or bind by name) to the interfaces of,
      the Work and Derivative Works thereof.

      "Contribution" shall mean any work of authorship, including
      the original version of the Work and any modifications or additions
      to that Work or Derivative Works thereof, that is intentionally
      submitted to Licensor for inclusion in the Work by the copyright owner
      or by an individual or Legal Entity authorized to submit on behalf of
      the copyright owner. For the purposes of this definition, "submitted"
      means any form of electronic, verbal, or written communication sent
      to the Licensor or its representatives, including but not limited to
      communication on electronic mailing lists, source code control systems,
      and issue tracking systems that are managed by, or on behalf of, the
      Licensor for the purpose of discussing and improving the Work, but
      excluding communication that is conspicuously marked or otherwise
      designated in writing by the copyright owner as "Not a Contribution."

      "Contributor" shall mean Licensor and any individual or Legal Entity
      on behalf of whom a Contribution has been received by Licensor and
      subsequently incorporated within the Work.

   2. Grant of Copyright License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      copyright license to reproduce, prepare Derivative Works of,
      publicly display, publicly perform, sublicense, and distribute the
      Work and such Derivative Works in Source or Object form.

   3. Grant of Patent License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      (except as stated in this section) patent license to make, have made,
      use, offer to sell, sell, import, and otherwise transfer the Work,
      where such license applies only to those patent claims licensable
      by such Contributor that are necessarily infringed by their
      Contribution(s) alone or by combination of their Contribution(s)
      with the Work to which such Contribution(s) was submitted. If You
      institute patent litigation against any entity (including a
      cross-claim or counterclaim in a lawsuit) alleging that the Work
      or a Contribution incorporated within the Work constitutes direct
      or contributory patent infringement, then any patent licenses
      granted to You under this License for that Work shall terminate
      as of the date such litigation is filed.

   4. Redistribution. You may reproduce and distribute copies of the
      Work or Derivative Works thereof in any medium, with or without
      modifications, and in Source or Object form, provided that You
      meet the following conditions:

      (a) You must give any other recipients of the Work or
          Derivative Works a copy of this License; and

      (b) You must cause any modified files to carry prominent notices
          stating that You changed the files; and

      (c) You must retain, in the Source form of any Derivative Works
          that You distribute, all copyright, patent, trademark, and
          attribution notices from the Source form of the Work,
          excluding those notices that do not pertain to any part of
          the Derivative Works; and

      (d) If the Work includes a "NOTICE" text file as part of its
          distribution, then any Derivative Works that You distribute must
          include a readable copy of the attribution notices contained
          within such NOTICE file, excluding those notices that do not
          pertain to any part of the Derivative Works, in at least one
          of the following places: within a NOTICE text file distributed
          as part of the Derivative Works; within the Source form or
          documentation, if provided along with the Derivative Works; or,
          within a display generated by the Derivative Works, if and
          wherever such third-party notices normally appear. The contents
          of the NOTICE file are for informational purposes only and
          do not modify the License. You may add Your own attribution
          notices within Derivative Works that You distribute, alongside
          or as an addendum to the NOTICE text from the Work, provided
          that such additional attribution notices cannot be construed
          as modifying the License.

      You may add Your own copyright statement to Your modifications and
      may provide additional or different license terms and conditions
      for use, reproduction, or distribution of Your modifications, or
      for any such Derivative Works as a whole, provided Your use,
      reproduction, and distribution of the Work otherwise complies with
      the conditions stated in this License.

   5. Submission of Contributions. Unless You explicitly state otherwise,
      any Contribution intentionally submitted for inclusion in the Work
      by You to the Licensor shall be under the terms and conditions of
      this License, without any additional terms or conditions.
      Notwithstanding the above, nothing herein shall supersede or modify
      the terms of any separate license agreement you may have executed
      with Licensor regarding such Contributions.

   6. Trademarks. This License does not grant permission to use the trade
      names, trademarks, service marks, or product names of the Licensor,
      except as required for reasonable and customary use in describing the
      origin of the Work and reproducing the content of the NOTICE file.

   7. Disclaimer of Warranty. Unless required by applicable law or
      agreed to in writing, Licensor provides the Work (and each
      Contributor provides its Contributions) on an "AS IS" BASIS,
      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
      implied, including, without limitation, any warranties or conditions
      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
      PARTICULAR PURPOSE. You are solely responsible for determining the
      appropriateness of using or redistributing the Work and assume any
      risks associated with Your exercise of permissions under this License.

   8. Limitation of Liability. In no event and under no legal theory,
      whether in tort (including negligence), contract, or otherwise,
      unless required by applicable law (such as deliberate and grossly
      negligent acts) or agreed to in writing, shall any Contributor be
      liable to You for damages, including any direct, indirect, special,
      incidental, or consequential damages of any character arising as a
      result of this License or out of the use or inability to use the
      Work (including but not limited to damages for loss of goodwill,
      work stoppage, computer failure or malfunction, or any and all
      other commercial damages or losses), even if such Contributor
      has been advised of the possibility of such damages.

   9. Accepting Warranty or Additional Liability. While redistributing
      the Work or Derivative Works thereof, You may choose to offer,
      and charge a fee for, acceptance of support, warranty, indemnity,
      or other liability obligations and/or rights consistent with this
      License. However, in accepting such obligations, You may act only
      on Your own behalf and on Your sole responsibility, not on behalf
      of any other Contributor, and only if You agree to indemnify,
      defend, and hold each Contributor harmless for any liability
      incurred by, or claims asserted against, such Contributor by reason
      of your accepting any such warranty or additional liability.

   END OF TERMS AND CONDITIONS

   APPENDIX: How to apply the Apache License to your work.

      To apply the Apache License to your work, attach the following
      boilerplate notice, with the fields enclosed by brackets "[]"
      replaced with your own identifying information. (Don't include
      the brackets!)  The text should be enclosed in the appropriate
      comment syntax for the file format. We also recommend that a
      file or class name and description of purpose be included on the
      same "printed page" as the copyright notice for easier
      identification within third-party archives.

   Copyright [yyyy] [name of copyright owner]

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.

```
</details>